<center><font size="4"><b>Technologia Mowy 2026
</b></font></center>
<br />
<center><font size="6"><b><u>Lab 3: Parametryzacja (ekstrakcja cech) sygnału mowy</u></b></font></center>
<br />
<br />

<center><b>Notebook version:</b> 0.7 (17.10.2026)</center>
<center>Copyright Marcin Witkowski, AGH, 2026</center>


# Cele ćwiczenia*

W ramach tego notatnika:

*   Zapoznasz się z interfejsami metod do ekstrakcji cech MFCC
*   Przeanalizujesz rozkłady cech
*   Zapoznasz się z nowoczesnymi metodami ekstrakcji tonu podstawowego
*   Zaimplementujesz podstawowe metody ekstrakcji tonu podstawowego



# Wstęp*

Pomimo niezwykle intensywnego rozwoju metod sztucznej inteligencji operujących na STFT bądź bezpośrednio na sygnale mowy w niektórych dziedzinach parametryzacja dalej stanowi ważny element systemu mający na celu ułatwienie modelowania lub analizy.
Najbardziej popularnym algorytmem, stanowiącym podstawę większości systemów generacji sprzed enkoderów neuronowych była ekstrakcja cech mel-cepstralnych (Mel-Frequency Cepstral Coefficients), czyli MFCC. Obecnie tendencja pokazuje odejście od przetwarzania wstępnego używając spektrogramu (lub jego lekko zmodyfikowanej wersji) lub wręcz rzeczywistych próbek sygnału.



# MFCC

## Demonstracja obliczania MFCC*

W notatniku użyte zostaną implementacje z pakietów:
 - [Librosa](https://librosa.org/doc/latest/index.html) (a konkretnie [librosa.feature.mfcc](https://librosa.org/doc/latest/generated/librosa.feature.mfcc.html#librosa.feature.mfcc))
 - [Python Speech Features](https://github.com/jameslyons/python_speech_features)

In [ ]:
# Funkcja pomocnicza do rysowania obrazów 2D
import matplotlib.pyplot as plt

def plot_2d(x,y,Z,title, ylab, xlab="Time [sec]"):
    fig = plt.figure(figsize=(12, 3))
    plt.style.use(['dark_background']) #--> dla osób korzystających z ciemnego motywu
    xmin,xmax = x.min(), x.max()
    ymin,ymax = y.min(), y.max()
    dy, = -np.diff([ymin,ymax])/(y.shape[0]-1)
    im1 = plt.imshow(Z, extent=[xmin, xmax, ymin+dy/2, ymax-dy/2],
                     aspect='auto', origin='lower', interpolation='none')
    plt.xlabel(xlab)
    plt.title(title)
    plt.ylabel(ylab)
    fig.colorbar(im1)

# Wczytaj pakiety umożliwiające obliczanie MFCC
import librosa
!pip install python_speech_features
import python_speech_features as psf
import numpy as np

In [ ]:
# Pobierz plik audio
!rm ./example.wav || true
!wget http://home.agh.edu.pl/~witkow/audio/TS1M1_7_.WAV -O example.wav

In [ ]:
# Wczytaj plik audio
from IPython.display import Audio
# import scipy.io.wavfile as wav
# fs, x = wav.read("example.wav")
x, fs = librosa.load("example.wav",sr=16000, mono=True)
plt.figure()
plt.plot(np.arange(0,len(x)/fs,1/fs),x)
plt.xlabel("Time [s]")
plt.title("Długość sygnału: {} sec".format(round(len(x)/fs,2)))
Audio(x,rate=fs)

In [ ]:
#Zdefiniuj parametry do obliczenia mfcc
winlen = 320    # Długość okna
winhop = 160    # Interwał pomiędzy oknami (przeskok)
n_fft = 512     # Liczba prążków w STFT
window='hamming'
n_filt = 30     # Liczba filtrów melowych
n_mfcc = 10     # Wyjściowa liczba wymiarów MFCC (cech)

In [ ]:
# STFT
from scipy import signal, fftpack
f, t, X = signal.stft(x, fs, nperseg=winlen, noverlap=winlen-winhop, nfft=n_fft, window=window, padded=False, boundary=None)

# Power spectrogram
PS = np.abs(X)**2
print("Liczba ramek to ",1 + np.floor((len(x)-winlen)/winhop))
print(np.shape(PS))
plot_2d(t,f,PS,"$|STFT|^2$","Frequency [Hz]")

In [ ]:
# Bank filtrów melowych
fbank =  librosa.filters.mel(sr=fs, n_fft=n_fft, n_mels=n_filt, norm=1)
print(np.shape(fbank))
n = 1
print(f"Suma wartości filtru dla {n}-tego filtru: {np.sum(fbank[n,])}")

upper_freq = 8000
f_idx = range(0,int(1+upper_freq/fs*n_fft))
plot_2d(np.arange(0,n_filt),f[f_idx],fbank.T[f_idx],"Filterbank","Frequency [Hz]", "Nr filtru")

plt.figure(figsize=(12, 3))
plt.plot(f[f_idx],fbank.T[f_idx])
plt.xlabel('Frequency [Hz]')
plt.show()

In [ ]:
# Mel Power Spectrogram
MPS = np.dot(fbank,PS)
print(np.shape(MPS))
plot_2d(t,np.arange(0,n_filt),MPS,"Mel Power Spectrogram","# Filter")

In [ ]:
# Logarithm
logMPS = 10*np.log10(MPS)
plot_2d(t,np.arange(0,n_filt),logMPS,"Log(Mel Power Spectrogram)","# Filter")

In [ ]:
# DCT
MFCC = fftpack.dct(logMPS, axis=0, type=2, norm='ortho')[:n_mfcc]
print(np.shape(MFCC))
id_mfcc = np.arange(0,n_mfcc) # Zamień 1 na 0 by uwzględnić c0 w cechach
plot_2d(t,id_mfcc, MFCC[id_mfcc],"MFCC","# Coeff.")

# Liftering (postprocessing)
lifter =  2 * n_mfcc
if lifter > 0:
    xx = np.arange(1,1 + n_mfcc)
    yy_lift = (1 + (lifter/2) * np.sin(np.pi * xx / lifter))[:, np.newaxis]
    plt.figure()
    plt.plot(xx,yy_lift)
    plt.xlabel("# Filter")
    plt.show()
    MFCC_lift = MFCC * yy_lift
else:
    MFCC_lift = MFCC

plot_2d(t,id_mfcc, MFCC_lift[id_mfcc],"Liftered MFCC","# Coeff.")

In [ ]:
# Porównanie Librosa vs Python Speech Features
# Librosa (from logMPS)
MFCC_librosa_spec = librosa.feature.mfcc(x=None, S=logMPS, sr=fs,
                                         n_mfcc=n_mfcc, lifter=lifter)
print(np.shape(MFCC_librosa_spec))
plot_2d(t,id_mfcc, MFCC_librosa_spec[id_mfcc], "Librosa MFCC (from logMPS)", "# Coeff.")

# Librosa (from x)
MFCC_librosa_x = librosa.feature.mfcc(y=x, S=None, sr=fs,
                                      n_mfcc=n_mfcc, lifter=lifter,
                                      n_fft=n_fft, hop_length=winhop,
                                      win_length=winlen, center=True,
                                      window=window)
print(np.shape(MFCC_librosa_x))
tt = np.arange(1,np.shape(MFCC_librosa_x)[1]-1)
plot_2d(t,id_mfcc, MFCC_librosa_x[1:n_mfcc, tt], "Librosa MFCC (from x)", "# Coeff.")

# Python speech features
MFCC_psf = psf.mfcc(x,samplerate=fs, winlen=winlen/fs, winstep=winhop/fs,
                    numcep=n_mfcc, nfilt=n_filt, nfft=n_fft, ceplifter=lifter,
                    preemph=0, winfunc=lambda M:np.hamming(M))
print(np.shape(MFCC_psf))
plot_2d(t,id_mfcc, MFCC_psf.T[1:n_mfcc,tt], "Transposed Python Speech Features MFCC", "# Coeff.")

Zwróć uwagę, że pomimo podobieństwa wizualnego macierzy cech mają one znacznie inne wartości numeryczne. Oznacza to, że próbując porównać cechy, które teoreczynie zostały obliczone tym samym algorytmem, ważne jest by mieć pewność, ze została użyta również ta sama implementacja. **W przeciwnym wypadku porównanie może być niemożliwe.**

In [ ]:
 # Porównanie wyników wywołania funkcji z domyślnymi wartościami:
 # Librosa (from x)
MFCC_librosa_x_default = librosa.feature.mfcc(y=x, S=None, sr=fs )
t,f =np.arange(0,np.shape(MFCC_librosa_x_default)[0]),np.arange(0,np.shape(MFCC_librosa_x_default)[1])
plot_2d(t,f, MFCC_librosa_x_default[1:n_mfcc], "Librosa MFCC (from x)", "# Coeff.")

# Python speech features
MFCC_psf_default = psf.mfcc(x,samplerate=fs)
t,f = np.arange(0,np.shape(MFCC_psf_default)[0]),np.arange(0,np.shape(MFCC_psf_default)[1])
plot_2d(t,f, (MFCC_psf_default.T)[1:n_mfcc], "Python Speech Features MFCC", "# Coeff.")

## Histogram*


Poniższy przykład demonstruje użycie funkcji `matplotlib.pyplot.hist()`.


In [ ]:
import numpy as np

d = np.random.normal(loc=15, scale=3, size=500)  # gen. dane o rozkładzie normalnym
mu = np.round(np.mean(d),2)
dev = np.round(np.std(d),2)

import matplotlib.pyplot as plt

# An "interface" to matplotlib.axes.Axes.hist() method
n, bins, patches = plt.hist(x=d, bins='auto', color='#0504aa',
                            alpha=0.7, rwidth=0.85)
plt.grid(axis='y', alpha=0.75)
plt.xlabel('Value')
plt.ylabel('Frequency')
plt.text(7, 45, rf'$\hat{{\mu}}={mu}, \hat{{\sigma}}={dev}$')
maxfreq = n.max()
# Set a clean upper y-axis limit.
plt.ylim(ymax=np.ceil(maxfreq / 10) * 10 if maxfreq % 10 else maxfreq + 10)

## Zadania

Ta sekcja zawiera 3 zadania oceniane odpowiednio na 0.2p, 0.3p i 1p.






### 1.1 (0.2p) Nagranie samogłosek


W ramach grupy 2 osoby (tej samej płci) powinny nagrać pliki WAV (fs=16kHz, 16bit, mono, czas około 2-3 sekundy) zawierające:
* głoskę /u/ o przedłużonej fonacji ("uuuuu...");
* głoskę /i/ o przedłużonej fonacji ("iiiiiiiiiiiiiii...").

Podczas nagrań należy nie modulować głosu, tak by uzyskać stały ton krtaniowy (nie musi być taki sam dla obu członkiń/członków grupy). Nagrywając, zadbaj o wysoki SNR, redukując pogłos i eliminując inne źródła dźwięku. Nagrania należy zapisać w katalogu współdzielonym i wczytać je w tym notatniku do zmiennych. Każda grupa powinna nagrać 4 pliki.

In [ ]:
# W tym bloku zamontuj dysk Google, przejdź do katalogu grupy i wczytaj nagrane pliki


### 1.2 (0.3p) MFCC oraz histogramy

W poniższym bloku kodu dla obu członkiń/członków grupy:
1. wyznacz i narysuj macierze MFCC (bez delt i delt-delt, skorzystaj z gotowych implementacji MFCC) oraz  
2. Na podstawie macierzy MFCC **wybierz parę cech** dla których wartości są znacznie różne dla różnych głosek (pierwsza cecha), wartości są porównywalne (druga cecha), a następnie za pomocą dostępnych funkcji wyznacz i narysuj (korzystając z `plt.subplot`)
   *  histogram pierwszej cechy dla głoski /u/
   *  histogram pierwszej cechy dla głoski /i/
   *  histogram drugiej cechy dla głoski /u/
   *  histogram drugiej cechy dla głoski /i/

In [ ]:
# Tu oblicz macierze MFCC dla obu głosek oraz narysuj histogramy cechy I i II


### 1.3 (1p) Analiza cech

1.3.1 (0.3p) Korzystając z [testu Shapiro-Wilk'a](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html) określ poziomy ufności dla których można odrzucić hipotezę zerową o normalności rozkładu każdej z cech. Następnie skomentuj otrzymane wyniki odpowiadając na pytania:

(1) Które cechy można uznać za opisane rozkładem normalnym (czyli takie dla których mozna użyć do porównania średnich i odchyleń standardowych)?

(2) Jak otrzymane wyników testów mają się do jakościowej oceny histogramów otrzymanych w zadaniu 1.2?


In [ ]:
# Tu zbadaj normalność rozkładów


<font color=red>Twoja odpowiedz do zadania 1.3.1</font>




1.3.2. (0.3p) Przeanalizuj średnie i odchylenia standardowe.

(1) Jakie są wartości średnie i odchylenia standardowe tych współczynników? Wartości liczbowe wydrukuj w sposób czytelny lub zamieść je w tabeli poniżej kodu. (**Porada:** Jeśli chcesz wstawić tabelę do notatnika, skorzystaj z dowolnego generatora tabel online, np. [www.tablesgenerator.com](https://www.tablesgenerator.com/markdown_tables) dla Markdown używanego w notatnikach a następnie przeklej wygenerowany tekst ponżej)

(2)* Dla każdej z cech wyznaczonych dla głosek U oraz I oblicz kryterium dyskryminatywne Fishera jako
$$ FDR(\textrm{c}_i) = \frac{\big(\mu_U(\textrm{c}_i) - \mu_I(\textrm{c}_i)\big)^2}{\sigma^2_U(\textrm{c}_i) + \sigma^2_I(\textrm{c}_i)}$$
gdzie $\textrm{c}_i$ wskazuje porównywaną cechę.

In [ ]:
# Tu wyznacz średnie i odchylenia standardowe cech oraz FDR dla obu głosek


1.3.3. (0.4p) Odpowiedz na pytanie:

**Ostatecznie Która z wybranych cech jest bardziej dyskryminatywna (umożliwia łatwiejsze rozróżnienie samogłosek)?**

Na podstawie histogramów oraz wyliczonych parametrów statystycznych w zadaniach 1.3.1 i 1.3.2 określ która cecha umożliwia łatwiejsze rozróżnienie **samogłosek** i opisz konkretny klasyfikator (wskaż precyzyjnie wartość/wartości progowe) jaki mógłby zostać użyty do rozróżniania głosek.

<font color=red>Twoja odpowiedz do zadania 1.3.3</font>

# Ekstrakcja F0



## Demonstracja metod ekstrakcji F0*

### Spektrogram + F0
Poniższy kod umożliwia dość eleganckie wyświetlanie spektrogramu z możliwością naniesienia konturu F0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa.display

def spec_with_f0(y,sr,f0,times,tit):
    D = librosa.amplitude_to_db(np.abs(librosa.stft(y)), ref=np.max)
    fig, ax = plt.subplots()
    img = librosa.display.specshow(D, x_axis='time', y_axis='log', ax=ax)
    ax.set(title=tit) #'pYIN fundamental frequency estimation')
    fig.colorbar(img, ax=ax, format="%+2.f dB")
    ax.plot(times, f0, label='f0', color='cyan', linewidth=3)
    ax.legend(loc='upper right')

### pYIN (2014)

Metoda pYIN[1] jest w istocie znacznie rozwiniętą metodą opartą o autokorelację (YIN [2]), gdzie wyjściowe estymaty F0 dla poszczególnych ramek są dodatkowo dekodowane za pomocą algorytmu Viterbiego, skąd przedrostek "p" od *probabilistic*.

Źródła:

[[1]](https://www.eecs.qmul.ac.uk/~simond/pub/2014/MauchDixon-PYIN-ICASSP2014.pdf) Mauch, Matthias, and Simon Dixon. “pYIN: A fundamental frequency estimator using probabilistic threshold distributions.” 2014 IEEE International Conference on Acoustics, Speech and Signal Processing (ICASSP). IEEE, 2014.

[[2]](http://audition.ens.fr/adc/pdf/2002_JASA_YIN.pdf) De Cheveigné, Alain, and Hideki Kawahara. “YIN, a fundamental frequency estimator for speech and music.” The Journal of the Acoustical Society of America 111.4 (2002): 1917-1930.



In [ ]:
import librosa
from IPython.display import Audio

y, sr = librosa.load(librosa.ex('trumpet'))
f0_pyin, voiced_flag, voiced_probs = librosa.pyin(y, fmin=100, fmax=1000)
# f0_pyin, voiced_flag, voiced_probs = librosa.pyin(y, fmin=librosa.note_to_hz('C2'), fmax=librosa.note_to_hz('C7'))
times = librosa.times_like(f0_pyin)

spec_with_f0(y,sr,f0_pyin,times,'pYIN')
Audio(data=y, rate=sr)

### CREPE (2018)

Metoda CREPE [1] oparta jest o estymację skwantowanych wartości F0 na podstawie konwolucyjnej sieci neuronowej operującej na 1024 próbkowych blokach sygnału próbkowanego z częstotliwością 16 *kHz*. Z racji swojej natury metoda zwraca skwantyzowane wartości centów. Cytując publikację:
"*The 360 pitch values are denoted as ¢1, ¢2, · · · , ¢360 and are selected so that they cover six octaves with 20-cent intervals between C1 and B7, corresponding to 32.70 Hz and 1975.5 Hz*"


Źródła: [[1]](https://arxiv.org/abs/1802.06182) Jong Wook Kim, Justin Salamon, Peter Li, Juan Pablo Bello, *CREPE: A Convolutional Representation for Pitch Estimation*, ICASSP 2018

In [ ]:
! pip install --upgrade tensorflow  # if you don't already have tensorflow >= 2.0.0
! pip install crepe

In [ ]:
import crepe
import librosa
y, sr = librosa.load(librosa.ex('trumpet'))
times_c, f0_crepe, confidence, activation = crepe.predict(y, sr, viterbi=True, step_size=50)
spec_with_f0(y,sr,f0_crepe,times_c,'CREPE')

### SwiftF0

In [ ]:
!pip install swift-f0

In [ ]:
from swift_f0 import SwiftF0, segment_notes, plot_pitch
import librosa
y, sr = librosa.load(librosa.ex('trumpet'))

model = SwiftF0()

result = model.detect(audio=y, sample_rate=sr, fmin=100, fmax=1000)
plot_pitch(result)

## Zadania

W ramach poniższych zadań określisz F0 analizując spektrogram oraz wykorzystując metody poznane na wykładzie i ćwiczeniach - metodę autokorelacyjną oraz cepstralną.

### 2.1. (0.5p) Oszacowanie F0 na podstawie spektrogramu

Korzystając z odpowiednio przeskalowanego spektrogramu oszacuj jakościowo wartość oczekiwaną tonu podstawowego dla jednego z plików audio z zadania 1. Długość okna dobierz tak by szacowanie wykonać z dokładnością +/- 5Hz. W rozwiązaniu przedstaw spektrogram o zadanej dokładnośći w dz. częstotliwości oraz odpowiednim zbliżeniu, tak by wartość częstotliwości można było łatwo oszacować z obrazu.

In [ ]:
# Tu wpisz kod i wyświetl fragment spektrogramu umożliwiający oszacowanie F0 z
# dokładnością +/- 5Hz.

### 2.2. (0.5p) Autokorelacja


Korzystając z metody autokorelacji wyznacz średnią wartość F0 dla całego pliku z zadania 1.1. W ramach implementacji **narysuj wykres funkcji autokorelacji w odpowiednim zakresie, z poziomą osią zawierającą częstotliwości tonu podstawowego.** Jeśli wykorzystana zostanie implementacja znaleziona w internecie lub wygenerowana za pomocą modelu językowego należy wskazać źródło w komentarzu, ale mocno wskazane jest wykonanie własnej implementacji w celach dydaktycznych. Czy otrzymany wynik pokrywa się z wartością oszacowaną w zadaniu 2.1? W przypadku różnych wartości omów jakie mogą być tego przyczyny.


In [ ]:
# Miejsce na kod do Zadania 2

### 2.3. (0.5p) Cepstrum

Korzystając z metody cepstralnej wyznacz wartość F0 dla całego pliku z zadania 1.1. W ramach implementacji **narysuj wykres cepstrum w odpowiednim zakresie i z poziomą osią zawierającą częstotliwości tonu podstawowego**. Jeśli wykorzystana zostanie implementacja znaleziona w internecie lub z wykorzystaniem modelu językowego należy wskazać źródło w komentarzu. Czy otrzymany wynik pokrywa się z wartością oszacowaną w zadaniu 2.1?


In [ ]:
# Miejsce na kod do Zadania 3

## Wnioski z Zadania 2

<font color=red> Miejsce na Twoje wnioski </color>

# Ocena
###<font color=orange>* Przed ostatecznym zapisem **usuń bloki zawierające gwiazdkę (*) w tytule** by ułatwić sprawdzanie prowadzącemu! </font>

### * 1 termin tego zadania upływa **4.11.2026 o godz. 13:00**. Po skończonej pracy należy zapisać plik opcją `File > Save and pin revision`. Prowadzący oceniając odtworzy ostanią wersję w repozytorium notatnika sprzed daty oddania i ta wersja będzie podlegać ocenie.